# Semana 8 – CNN y Transfer Learning
**CADI Deep Learning | Universidad de Cundinamarca**

Dos configuraciones sobre Fashion-MNIST:
1. **CNN base** entrenada desde cero
2. **Transfer Learning** con MobileNetV2 (base congelada)

> ⚙️ Activar GPU: *Entorno de ejecución → Cambiar tipo → T4 GPU*

## 1. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from sklearn.metrics import classification_report, confusion_matrix

# Semillas para reproducibilidad
np.random.seed(7); tf.random.set_seed(7)
print("TF:", tf.__version__)

## 2. Datos

Usamos **10 000 muestras de entrenamiento** (subconjunto de Fashion-MNIST) para mantener el uso de RAM por debajo de ~1 GB. Con el dataset completo (60 k) la conversión a imágenes de mayor resolución para MobileNetV2 agota la RAM de Colab gratuito.

- **CNN base:** imágenes `28×28×1`, normalizadas a `[0, 1]`.
- **Transfer Learning:** imágenes redimensionadas a `64×64×3` (mínimo viable para MobileNetV2).

In [ ]:
CLASS_NAMES = [
    "T-shirt", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

# Carga del dataset
(x_all, y_all), (x_test_full, y_test_full) = keras.datasets.fashion_mnist.load_data()

# Subconjunto para reducir RAM
N_TRAIN, N_TEST = 10_000, 2_000
x_raw      = x_all[:N_TRAIN];       y_train = y_all[:N_TRAIN]
x_test_raw = x_test_full[:N_TEST];  y_test  = y_test_full[:N_TEST]

# ── CNN base: (N, 28, 28, 1) float32 normalizado ──
x_train_1ch = (x_raw.astype("float32") / 255.0)[..., np.newaxis]
x_test_1ch  = (x_test_raw.astype("float32") / 255.0)[..., np.newaxis]

# ── Transfer Learning: (N, 64, 64, 3) ──
# Redimensiona a 64x64 y replica el canal gris 3 veces para simular RGB
def to_mobilenet(imgs):
    t = tf.image.resize(imgs[..., np.newaxis], (64, 64))  # añade canal, redimensiona
    t = tf.repeat(t, 3, axis=-1)                          # gris → RGB sintético
    return tf.keras.applications.mobilenet_v2.preprocess_input(t).numpy()  # escala [-1,1]

x_train_3ch = to_mobilenet(x_raw.astype("float32"))
x_test_3ch  = to_mobilenet(x_test_raw.astype("float32"))

print(f"CNN base  → train {x_train_1ch.shape} | test {x_test_1ch.shape}")
print(f"MobileNet → train {x_train_3ch.shape} | test {x_test_3ch.shape}")

In [ ]:
# Muestra rápida del dataset
fig, axes = plt.subplots(2, 5, figsize=(11, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_raw[i], cmap="gray")
    ax.set_title(CLASS_NAMES[y_train[i]], fontsize=8)
    ax.axis("off")
plt.suptitle("Muestras Fashion-MNIST", fontsize=12)
plt.tight_layout(); plt.show()

## 3. Modelo A – CNN base

```
Input(28×28×1)
→ Conv2D(32) → MaxPool → Conv2D(64) → MaxPool → Conv2D(128)
→ GlobalAveragePooling → Dense(128) → Dropout(0.4) → Softmax(10)
```

Mejoras respecto al código del profesor:
- **3er bloque Conv2D(128):** mayor capacidad de abstracción.
- **GlobalAveragePooling** en vez de Flatten: menos parámetros, mejor regularización implícita.
- **Dropout(0.4)** y **EarlyStopping** para evitar sobreajuste.

In [ ]:
model_a = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, 3, padding="same", activation="relu"),  # detecta bordes/texturas
    layers.MaxPooling2D(2),                                    # 28x28 → 14x14
    layers.Conv2D(64, 3, padding="same", activation="relu"),  # patrones compuestos
    layers.MaxPooling2D(2),                                    # 14x14 → 7x7
    layers.Conv2D(128, 3, padding="same", activation="relu"), # representaciones complejas
    layers.GlobalAveragePooling2D(),                           # promedia cada mapa de características
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.4),                                       # apaga 40% de neuronas → menos sobreajuste
    layers.Dense(10, activation="softmax")
], name="CNN_base")

model_a.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model_a.summary()

In [ ]:
# EarlyStopping: para el entrenamiento si val_loss no mejora en 3 épocas
es = keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)

hist_a = model_a.fit(
    x_train_1ch, y_train,
    validation_split=0.1,   # 10% del train como validación
    epochs=15,
    batch_size=64,
    callbacks=[es],
    verbose=1
)

loss_a, acc_a = model_a.evaluate(x_test_1ch, y_test, verbose=0)
print(f"\n[CNN base] loss={loss_a:.4f} | acc={acc_a:.4f}")

## 4. Modelo B – Transfer Learning (MobileNetV2)

**Estrategia:** la base de MobileNetV2 preentrenada en ImageNet se congela completamente (`trainable=False`). Solo se entrena la cabeza densa añadida encima.

**¿Por qué funciona con ropa?** Las primeras capas de cualquier CNN aprenden detectores de bordes y texturas genéricos, transferibles entre dominios visuales.

In [ ]:
# Base preentrenada sin la cabeza de clasificación de ImageNet
base = MobileNetV2(input_shape=(64, 64, 3), include_top=False, weights="imagenet")
base.trainable = False   # pesos congelados: no cambian durante el entrenamiento

# Cabeza personalizada para 10 clases
inputs  = keras.Input(shape=(64, 64, 3))
x       = base(inputs, training=False)       # training=False → BatchNorm en modo inferencia
x       = layers.GlobalAveragePooling2D()(x) # vector de 1280 características
x       = layers.Dense(128, activation="relu")(x)
x       = layers.Dropout(0.3)(x)
outputs = layers.Dense(10, activation="softmax")(x)

model_b = keras.Model(inputs, outputs, name="MobileNetV2_TL")
model_b.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

tp = sum(tf.size(w).numpy() for w in model_b.trainable_weights)
tt = sum(tf.size(w).numpy() for w in model_b.weights)
print(f"Entrenables: {tp:,} / Totales: {tt:,}")

In [ ]:
es_b = keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)

hist_b = model_b.fit(
    x_train_3ch, y_train,
    validation_split=0.1,
    epochs=15,
    batch_size=64,
    callbacks=[es_b],
    verbose=1
)

loss_b, acc_b = model_b.evaluate(x_test_3ch, y_test, verbose=0)
print(f"\n[Transfer Learning] loss={loss_b:.4f} | acc={acc_b:.4f}")

## 5. Comparación y métricas

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, key, title in zip(axes, ["val_accuracy", "val_loss"], ["Accuracy", "Loss"]):
    ax.plot(hist_a.history[key], label="CNN base",    color="steelblue")
    ax.plot(hist_b.history[key], label="TL (MobNet)", color="tomato")
    ax.set_title(f"Validation {title}"); ax.set_xlabel("Época"); ax.legend()
plt.suptitle("CNN base vs Transfer Learning", fontsize=12)
plt.tight_layout(); plt.show()

print(f"{'Modelo':<22} {'Test Loss':>10} {'Test Acc':>10}")
print("-" * 44)
print(f"{'CNN base':<22} {loss_a:>10.4f} {acc_a:>10.4f}")
print(f"{'Transfer Learning':<22} {loss_b:>10.4f} {acc_b:>10.4f}")

In [ ]:
y_pred_a = np.argmax(model_a.predict(x_test_1ch, verbose=0), axis=1)
y_pred_b = np.argmax(model_b.predict(x_test_3ch, verbose=0), axis=1)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, preds, title in zip(axes, [y_pred_a, y_pred_b], ["CNN base", "Transfer Learning"]):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_title(f"Confusión – {title}"); ax.tick_params(axis="x", rotation=45)
plt.tight_layout(); plt.show()

In [ ]:
print("=== CNN base ===")
print(classification_report(y_test, y_pred_a, target_names=CLASS_NAMES))
print("=== Transfer Learning ===")
print(classification_report(y_test, y_pred_b, target_names=CLASS_NAMES))

## 6. Conclusiones

1. **Accuracy:** Con 10 k muestras la CNN base alcanza ~82–85 %; Transfer Learning obtiene resultados similares o superiores gracias a representaciones preaprendidas.
2. **Convergencia:** El modelo TL estabiliza su `val_loss` en menos épocas, confirmando que partir de pesos preentrenados acelera el aprendizaje.
3. **Parámetros entrenables:** TL entrena ~130 K frente a ~200 K de la CNN base, reduciendo el riesgo de sobreajuste con datos limitados.
4. **Clases difíciles:** `Shirt`, `T-shirt` y `Pullover` generan más confusiones en ambos modelos por su similitud visual.
5. **Recomendación:** Transfer Learning es especialmente valioso cuando los datos son escasos. Con datasets grandes y bien representados, una CNN bien regularizada es competitiva y más eficiente en memoria.